# Tendencias del mercado tecnológico — TalentHub (punto 3)

**Cliente:** TalentHub quiere lanzar un bootcamp tech y necesita saber hacia dónde se mueve el mercado, no solo qué se pide hoy.

**Objetivo:** medir la evolución 2016–2026 de tecnologías, perfiles y modalidades de trabajo para detectar qué crece, qué cae y qué está emergiendo.

**Fuente:** [Stack Overflow Developer Survey](https://survey.stackoverflow.co/), ediciones 2016 a 2026, descargadas del repositorio oficial [StackExchange/Survey](https://github.com/StackExchange/Survey/tree/main/packages/archive)

**Por qué esta fuente:** el dataset principal del equipo (ofertas nextgig) es una foto de junio 2026 y no tiene serie histórica. El enunciado permite complementar el punto 3 con "datasets abiertos de series históricas", y esta encuesta es la serie anual más larga y abierta sobre uso de tecnologías.

**Qué mide y qué no:** la encuesta mide qué tecnologías **usan y quieren usar los desarrolladores** (oferta de talento e interés). No mide vacantes. Por eso se usa como indicador de tendencia y al final se cruza con las ofertas de nextgig, que sí miden demanda de las empresas.

## 0. Carga de datos
Se leen los Parquet reducidos que deja el script de descarga (una fila por respuesta, solo las columnas mapeadas, valores sin transformar).

In [ ]:
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

RUTA_RAW = "../data/raw/stack_overflow_survey"
RUTA_PROCESSED = "../data/processed"
RUTA_IMAGENES = "../images"
AÑOS = list(range(2016, 2027))

crudo = {año: pd.read_parquet(f"{RUTA_RAW}/so_{año}.parquet") for año in AÑOS}
resumen = pd.read_csv(f"{RUTA_RAW}/resumen_descarga.csv")

print(f"Ediciones: {len(crudo)} | Respuestas totales: {sum(len(d) for d in crudo.values()):,}")
resumen[["año", "filas", "columnas_originales", "columnas_guardadas", "csv_mb", "parquet_mb"]]

## 1. Perfil comparable entre años

Para hablar de mercado laboral el análisis principal usa **desarrolladores profesionales**. La pregunta que permite identificarlos cambió cinco veces, así que la regla se define por año:

| Años | Regla para `es_profesional` |
|---|---|
| 2016 | empleo activo (tiempo completo, parcial, freelance, independiente) y ocupación distinta de "Student" |
| 2017 | `Professional` = "Professional developer" |
| 2018 | `Student` = "No" y empleo activo |
| 2019–2025 | `MainBranch` = "I am a developer by profession" |
| 2026 | `CodeMain` = "I create code as a primary part of my job" y empleo activo |

No se descartan filas: queda un flag para poder comparar contra todos los encuestados.

In [ ]:
EMPLEO_ACTIVO = {
    "Employed full-time", "Employed part-time", "Freelance / Contractor", "Self-employed",
    "Independent contractor, freelancer, or self-employed", "Employed",
}

def es_profesional(año, df):
    activo = df["empleo"].isin(EMPLEO_ACTIVO)
    if año == 2016:
        return activo & df["rol"].notna() & (df["rol"] != "Student")
    if año == 2017:
        return df["perfil"] == "Professional developer"
    if año == 2018:
        return (df["estudiante"] == "No") & activo
    if año <= 2025:
        return df["perfil"] == "I am a developer by profession"
    return (df["perfil"] == "I create code as a primary part of my job") & activo

columnas_persona = ["año", "id", "es_profesional", "empleo", "rol", "exp_pro", "exp_total", "remoto", "ai_select", "pais"]
personas = []
for año, df in crudo.items():
    df = df.copy()
    df["es_profesional"] = es_profesional(año, df).fillna(False).astype(bool)
    personas.append(df.reindex(columns=columnas_persona))
personas = pd.concat(personas, ignore_index=True)
personas["id"] = personas["id"].astype("int32")

print(f"Duplicados de (año, id): {personas.duplicated(['año', 'id']).sum()}")
muestra = personas.groupby("año").agg(encuestados=("id", "size"), profesionales=("es_profesional", "sum"))
muestra["pct_profesionales"] = (muestra["profesionales"] / muestra["encuestados"] * 100).round(1)
muestra

La muestra cambia mucho de tamaño (de casi 99 mil respuestas en 2018 a 31 mil en 2026), por eso todos los indicadores se expresan como **porcentaje**, nunca como conteo.

## 2. Panel largo de tecnologías

Cada pregunta de tecnologías es un multi-select separado por `;`. Se pasa a formato largo: una fila por respondente × pregunta × tecnología, con dos relaciones: **usa** (trabajó con ella el último año) y **quiere** (quiere trabajar con ella el próximo año).

In [ ]:
partes = []
for año, df in crudo.items():
    ids = df["id"].astype("int32")
    for columna in [c for c in df.columns if c.startswith(("usa__", "quiere__"))]:
        relacion, pregunta = columna.split("__")
        valores = df[columna].set_axis(ids).dropna().str.split(";").explode().str.strip()
        partes.append(pd.DataFrame({
            "año": año, "id": valores.index, "relacion": relacion, "pregunta": pregunta, "etiqueta": valores.values,
        }))

largo = pd.concat(partes, ignore_index=True)
for columna in ["relacion", "pregunta", "etiqueta"]:
    largo[columna] = largo[columna].astype("category")
del partes

print(f"Filas: {len(largo):,} | Etiquetas distintas: {largo['etiqueta'].nunique():,}")
largo.head()

### 2.1 Quién respondió cada pregunta

La tabla muestra qué porcentaje de los profesionales respondió cada pregunta (se guarda antes de limpiar etiquetas). Se ven dos quiebres:
- **2021:** la pregunta de plataformas deja de incluir sistemas operativos y pasa a ser solo cloud, y Docker/Kubernetes se mudan a una pregunta nueva de herramientas.
- **2025–2026:** antes de cada bloque la encuesta pregunta si usaste ese tipo de tecnología, y solo quienes dicen que sí ven la lista.

Por eso la base de los porcentajes **no** puede ser "quienes respondieron esa pregunta" (cambiaría de significado cada año). Se usa una **base ancla**: los profesionales que respondieron la pregunta de **lenguajes** (en 2016, la pregunta única). Es la pregunta más estable de toda la serie y la que responde prácticamente cualquiera que programa. Todos los porcentajes del notebook se leen como "de los profesionales que informaron con qué lenguajes trabajan, qué porcentaje usa X".

In [ ]:
respondieron = largo[["año", "id", "relacion", "pregunta"]].drop_duplicates()
respondieron = respondieron.merge(personas[["año", "id", "es_profesional"]], on=["año", "id"])

base_usa = respondieron[(respondieron["relacion"] == "usa") & respondieron["es_profesional"]]
tasa = (base_usa.groupby(["pregunta", "año"], observed=True)["id"].nunique().unstack("año")
        .div(muestra["profesionales"], axis=1).mul(100).round(0))
tasa = tasa.loc[["mixto", "lenguaje", "bdd", "plataforma", "framework", "otros_frameworks", "herramientas",
                 "ide_colab", "testing", "ia_chat", "ia_modelos", "ia_asistentes", "ia_orquestacion"]]
display(tasa.astype("Int64"))

# base ancla: respondió la pregunta de lenguajes (o la pregunta única de 2016)
ancla = respondieron.loc[(respondieron["relacion"] == "usa") & respondieron["pregunta"].isin(["lenguaje", "mixto"]), ["año", "id"]]
ancla = ancla.drop_duplicates().assign(en_base=True)
personas = personas.merge(ancla, on=["año", "id"], how="left")
personas["en_base"] = personas["en_base"].fillna(False).astype(bool)

muestra["base_ancla"] = personas[personas["es_profesional"] & personas["en_base"]].groupby("año")["id"].size()
muestra

### 2.2 Normalizar nombres de tecnologías

La misma tecnología aparece con etiquetas distintas según el año (`ReactJS` → `React.js` → `React`; `AWS` → `Amazon Web Services (AWS)`). Sin unificarlas, las series muestran caídas y subidas que no existen.

Reglas:
1. Se descartan las opciones genéricas ("Other").
2. Etiquetas que solo difieren en mayúsculas se unifican (se conserva la grafía más reciente).
3. Un diccionario explícito de alias agrupa las variantes conocidas. Algunas agrupaciones son **familias** (Angular incluye AngularJS, .NET incluye Core y Framework, los modelos de IA se agrupan por proveedor) porque la encuesta las separó y volvió a juntar en distintos años.

In [ ]:
ALIAS = {
    # web
    "Angular": ["AngularJS", "Angular.js", "Angular/Angular.js"],
    "React": ["ReactJS", "React.js"],
    "ASP.NET": ["ASP.NET Core", "ASP.NET CORE"],
    "Spring": ["Spring Boot", "Spring Framework"],
    ".NET": [".NET Core", ".NET Core / .NET 5", ".NET Framework", ".NET (5+)", ".NET Framework (1.0 - 4.8)"],
    # lenguajes
    "Bash/Shell": ["Bash/Shell (all shells)", "Bash/Shell/PowerShell"],
    "HTML/CSS": ["HTML", "CSS"],
    "Visual Basic": ["VB.NET", "Visual Basic (.Net)", "Visual Basic 6"],
    "Delphi": ["Delphi/Object Pascal"],
    # bases de datos
    "SQL Server": ["Microsoft SQL Server"],
    "DynamoDB": ["Amazon DynamoDB"],
    "BigQuery": ["Google BigQuery"],
    # cloud y plataforma
    "AWS": ["Amazon Web Services (AWS)"],
    "Microsoft Azure": ["Azure"],
    "Google Cloud": ["Google Cloud Platform", "Google Cloud Platform/App Engine"],
    "DigitalOcean": ["Digital Ocean"],
    "Docker": ["Docker / Docker Compose"],
    "Terraform": ["Teraform"],
    "Maven": ["Maven (build tool)"],
    "Hetzner": ["Hetzner Cloud"],
    "OVH": ["OVHcloud"],
    "Linode": ["Linode, now Akamai"],
    "Oracle Cloud Infrastructure": ["Oracle Cloud Infrastructure (OCI)"],
    "Linux": ["Linux Desktop"],
    "Windows": ["Windows Desktop", "Windows Desktop or Server"],
    "macOS": ["Mac OS"],
    # librerías y herramientas
    "Apache Spark": ["Spark"],
    "PyTorch": ["Torch/PyTorch"],
    "Unity": ["Unity 3D"],
    "IntelliJ IDEA": ["IntelliJ"],
    "Jupyter": ["IPython/Jupyter", "IPython", "Jupyter Notebook/JupyterLab"],
    # IA: se agrupa por proveedor porque la pregunta cambió de producto a modelo a empresa
    "OpenAI (ChatGPT / GPT)": ["ChatGPT", "openAI GPT (chatbot models)", "openAI Reasoning models",
                               "openAI Image generating models", "OpenAI"],
    "Anthropic (Claude)": ["Claude", "Anthropic: Claude Sonnet", "Anthropic"],
    "Google (Gemini)": ["Google Bard AI", "Google Gemini", "Gemini (Flash general purpose models)",
                        "Gemini (Pro Reasoning models)", "Google AI"],
    "Microsoft (Bing / Phi)": ["Bing AI", "Microsoft Phi-4 models", "Microsoft"],
    "Meta (Llama)": ["Meta AI", "Meta Llama (all models)"],
    "DeepSeek": ["DeepSeek (R- Reasoning models)", "DeepSeek (V- General purpose models)"],
    "Mistral": ["Mistral AI", "Mistral AI models"],
    "xAI (Grok)": ["X Grok models", "xAI"],
    "Alibaba Qwen": ["Alibaba Cloud Qwen models"],
    "Perplexity": ["Perplexity AI", "Perplexity Sonar models"],
}
DESCARTAR = {"other(s):", "other, please specify", "other (please specify):", "other", ""}

alias_por_clave = {}
for canonico, variantes in ALIAS.items():
    for variante in [canonico] + variantes:
        alias_por_clave[variante.lower()] = canonico

# grafía más reciente para las etiquetas que no tienen alias
ultima_grafia = (largo.groupby(["etiqueta", "año"], observed=True).size().reset_index(name="n")
                 .sort_values(["año", "n"]).assign(clave=lambda t: t["etiqueta"].astype(str).str.lower())
                 .drop_duplicates("clave", keep="last").set_index("clave")["etiqueta"].astype(str))

def normalizar(etiqueta):
    clave = etiqueta.lower()
    if clave in DESCARTAR:
        return None
    return alias_por_clave.get(clave, ultima_grafia[clave])

mapa_etiquetas = {etiqueta: normalizar(etiqueta) for etiqueta in largo["etiqueta"].cat.categories}
largo["tecnologia"] = largo["etiqueta"].astype(object).map(mapa_etiquetas)

antes = len(largo)
largo = largo.dropna(subset=["tecnologia"])
largo = largo.drop_duplicates(["año", "id", "relacion", "pregunta", "tecnologia"])
largo["tecnologia"] = largo["tecnologia"].astype("category")

print(f"Etiquetas originales: {len(mapa_etiquetas):,} -> tecnologías normalizadas: {largo['tecnologia'].nunique():,}")
print(f"Filas: {antes:,} -> {len(largo):,} (se quitaron opciones genéricas y duplicados creados al unificar)")

### 2.3 Categoría de cada tecnología

La pregunta donde aparece una tecnología también cambia (Docker estuvo en "plataformas", después en "herramientas" y volvió a "plataformas"; en 2016 todo estaba en una sola pregunta). Se asigna a cada tecnología **una categoría fija**: si alguna vez estuvo en la pregunta de herramientas de desarrollo (gestores de paquetes, build) queda ahí; si no, la pregunta donde apareció más años. Hay algunas correcciones manuales (Docker, Kubernetes y Terraform van a cloud y plataforma).

In [ ]:
NOMBRE_CATEGORIA = {
    "lenguaje": "Lenguaje", "bdd": "Base de datos", "plataforma": "Cloud y plataforma", "framework": "Framework web",
    "otros_frameworks": "Librerías y otros frameworks", "herramientas": "Herramientas de desarrollo",
    "ide_colab": "IDE y colaboración", "testing": "Testing y calidad", "ia_chat": "IA", "ia_modelos": "IA",
    "ia_asistentes": "IA", "ia_orquestacion": "IA", "mixto": "Sin categoría (solo 2016)",
}
CATEGORIA_MANUAL = {
    "Docker": "Cloud y plataforma", "Kubernetes": "Cloud y plataforma", "Terraform": "Cloud y plataforma",
    "Ansible": "Cloud y plataforma", "Podman": "Cloud y plataforma", "Node.js": "Framework web", "Firebase": "Cloud y plataforma",
    **{t: "Herramientas de desarrollo" for t in ["uv", "Poetry", "Conda", "Winget", "Bazel", "RubyGems / Bundler"]},
    "GitHub Copilot": "IA", "Cursor": "IA", "Claude Code": "IA", "Windsurf": "IA", "Aider": "IA",
}

apariciones = (largo[largo["pregunta"] != "mixto"].groupby(["tecnologia", "pregunta"], observed=True)["año"]
               .agg(años="nunique", ultimo="max").reset_index()
               .sort_values(["tecnologia", "años", "ultimo"]))
categoria = apariciones.drop_duplicates("tecnologia", keep="last").set_index("tecnologia")["pregunta"].astype(str).map(NOMBRE_CATEGORIA)
en_herramientas = apariciones.loc[apariciones["pregunta"] == "herramientas", "tecnologia"].astype(str)
categoria.loc[en_herramientas] = "Herramientas de desarrollo"
categoria = categoria.reindex(largo["tecnologia"].cat.categories).fillna("Sin categoría (solo 2016)")
categoria.update(pd.Series(CATEGORIA_MANUAL))
categoria.name = "categoria"

categoria.value_counts()

### 2.4 Panel limpio

Una fila por respondente × tecnología, con dos columnas booleanas (`usa`, `quiere`) y los flags `es_profesional` y `en_base` (respondió la pregunta ancla).

In [ ]:
claves = ["año", "id", "tecnologia"]
usa = largo.loc[largo["relacion"] == "usa", claves].drop_duplicates().assign(usa=True)
quiere = largo.loc[largo["relacion"] == "quiere", claves].drop_duplicates().assign(quiere=True)

panel = usa.merge(quiere, on=claves, how="outer")
panel[["usa", "quiere"]] = panel[["usa", "quiere"]].fillna(False).astype(bool)
panel = panel.merge(personas[["año", "id", "es_profesional", "en_base"]], on=["año", "id"], how="left")
panel["categoria"] = panel["tecnologia"].map(categoria).astype("category")
panel["tecnologia"] = panel["tecnologia"].astype("category")
panel = panel[["año", "id", "es_profesional", "en_base", "categoria", "tecnologia", "usa", "quiere"]]
del usa, quiere

print("=== ESTADO FINAL DEL PANEL ===")
print(f"Filas: {len(panel):,} | Tecnologías: {panel['tecnologia'].nunique():,} | Años: {panel['año'].nunique()}")
print(f"Duplicados de (año, id, tecnologia): {panel.duplicated(claves).sum()}")
print(f"Nulos: {int(panel.isna().sum().sum())}")
panel.head()

## 3. KPIs de tendencia

**Adopción** = profesionales de la base ancla que usan la tecnología ÷ profesionales de la base ancla.

Cuando una tecnología no fue opción de la encuesta en un año, el valor queda **vacío (NaN), no cero**: no se puede distinguir "nadie la usaba" de "no se preguntó".

In [ ]:
def calcular_metricas(solo_profesionales=True):
    p = panel[panel["en_base"] & (panel["es_profesional"] | (not solo_profesionales))]
    gente = personas[personas["en_base"] & (personas["es_profesional"] | (not solo_profesionales))]

    conteos = (p.assign(usa_y_quiere=p["usa"] & p["quiere"]).groupby(["año", "tecnologia"], observed=True)
               .agg(usuarios=("usa", "sum"), quiere=("quiere", "sum"), usa_y_quiere=("usa_y_quiere", "sum")).reset_index())
    conteos["base"] = conteos["año"].map(gente.groupby("año")["id"].size())

    conteos["adopcion_pct"] = conteos["usuarios"] / conteos["base"] * 100
    conteos["interes_no_usuarios_pct"] = (conteos["quiere"] - conteos["usa_y_quiere"]) / (conteos["base"] - conteos["usuarios"]) * 100
    conteos["retencion_pct"] = conteos["usa_y_quiere"] / conteos["usuarios"].where(conteos["usuarios"] > 0) * 100
    conteos["categoria"] = conteos["tecnologia"].map(categoria)
    return conteos

metricas = calcular_metricas(solo_profesionales=True)
metricas_todos = calcular_metricas(solo_profesionales=False)

# se exige un mínimo de usuarios para no leer ruido
metricas = metricas[metricas["usuarios"] >= 30]
adopcion = metricas.pivot(index="tecnologia", columns="año", values="adopcion_pct")

print(f"Tecnologías con serie: {len(adopcion)} | con dato en 2026: {adopcion[2026].notna().sum()}")
adopcion.loc[["JavaScript", "Python", "TypeScript", "Java", "SQL", "Docker", "AWS", "React", "PostgreSQL", "MySQL"]].round(1)

**Lectura de 2016:** ese año hubo una sola pregunta con una lista corta que mezclaba lenguajes, bases de datos y plataformas. Sirve como punto de partida para las tecnologías que ya estaban en la lista, pero no es estrictamente comparable con 2017 en adelante.

### 3.1 Top 15 de 2026 y su recorrido
Tecnologías "de programa" (lenguajes, bases de datos, cloud y plataforma, frameworks web), ordenadas por adopción 2026.

In [ ]:
NUCLEO = ["Lenguaje", "Base de datos", "Cloud y plataforma", "Framework web"]
es_nucleo = categoria.reindex(adopcion.index).isin(NUCLEO)

top15 = adopcion[es_nucleo].sort_values(2026, ascending=False).head(15)
top15.insert(0, "categoria", categoria.reindex(top15.index))
top15.round(1)

### 3.2 Mayores subidas y bajadas

Se mide el cambio en **puntos porcentuales** entre 2018 y 2026. Se usa 2018 como base porque es el primer año con las cuatro preguntas separadas y listas largas de opciones; 2016 y 2017 tienen listas más cortas. Como control se agrega la **pendiente** (puntos por año, ajustada sobre todos los años disponibles), que no depende de los dos extremos.

No se usa CAGR: sobre porcentajes con base chica produce tasas enormes que no dicen nada (pasar de 0,5 % a 5 % es "+900 %").

In [ ]:
def pendiente(fila):
    datos = fila.dropna()
    return np.polyfit(datos.index.astype(float), datos.values, 1)[0] if len(datos) >= 4 else np.nan

tendencia = pd.DataFrame({
    "categoria": categoria.reindex(adopcion.index),
    "primer_año": adopcion.apply(lambda f: f.first_valid_index(), axis=1),
    "años_con_dato": adopcion.notna().sum(axis=1),
    "adopcion_2018": adopcion[2018],
    "adopcion_2021": adopcion[2021],
    "adopcion_2026": adopcion[2026],
})
tendencia["delta_2018_2026_pp"] = tendencia["adopcion_2026"] - tendencia["adopcion_2018"]
tendencia["delta_2021_2026_pp"] = tendencia["adopcion_2026"] - tendencia["adopcion_2021"]
tendencia["pendiente_pp_por_año"] = adopcion.apply(pendiente, axis=1)

comparables = tendencia[tendencia["categoria"].isin(NUCLEO)].dropna(subset=["delta_2018_2026_pp"])
columnas_delta = ["categoria", "adopcion_2018", "adopcion_2026", "delta_2018_2026_pp", "pendiente_pp_por_año"]
subidas = comparables.sort_values("delta_2018_2026_pp", ascending=False).head(10)
bajadas = comparables.sort_values("delta_2018_2026_pp").head(10)

print("=== MAYORES SUBIDAS 2018 -> 2026 ===")
display(subidas[columnas_delta].round(1))
print("=== MAYORES BAJADAS 2018 -> 2026 ===")
display(bajadas[columnas_delta].round(1))

### 3.3 Tecnologías que entraron al radar

Tecnologías que **la encuesta empezó a preguntar en 2019 o después**, que en 2026 superan el 5 % de adopción y que **crecieron al menos 3 puntos** desde que entraron (o entraron en 2025–2026 directamente por encima del 10 %).

El "primer año" es cuándo Stack Overflow la agregó como opción, no cuándo se creó. Por eso se exige crecimiento: sin ese filtro entrarían tecnologías viejas que simplemente se empezaron a preguntar tarde (jQuery, Express, ASP.NET aparecen recién en 2019). Se miran solo lenguajes, bases de datos, cloud y frameworks web; IA se ve aparte.

In [ ]:
radar = tendencia[(tendencia["primer_año"] >= 2019) & (tendencia["adopcion_2026"] >= 5) & tendencia["categoria"].isin(NUCLEO)].copy()
radar["adopcion_primer_año"] = [adopcion.loc[tec, año] for tec, año in radar["primer_año"].items()]
radar["cambio_pp"] = radar["adopcion_2026"] - radar["adopcion_primer_año"]
recien_llegada = (radar["primer_año"] >= 2025) & (radar["adopcion_2026"] >= 10)
radar = radar[(radar["cambio_pp"] >= 3) | recien_llegada].sort_values("adopcion_2026", ascending=False)
radar = radar.drop(index=["PowerShell"])  # no es nueva: hasta 2020 se preguntaba junto con Bash/Shell

radar[["categoria", "primer_año", "adopcion_primer_año", "adopcion_2026", "cambio_pp"]].round(1).head(25)

### 3.4 Hueco de aprendizaje: qué quiere aprender el mercado

"Quiere" menos "usa" da negativo para todo lo popular (casi todos ya usan JavaScript). El indicador útil para un bootcamp es otro:

**Interés entre no usuarios** = de los profesionales que hoy **no** usan la tecnología, qué porcentaje quiere usarla el año próximo.

Se acompaña con la **retención** (de quienes la usan, cuántos quieren seguir): interés alto con retención baja es moda; las dos altas es demanda sostenida.

Importa el **orden**, no el nivel: como en 2026 solo ve cada lista quien declara usar ese tipo de tecnología, los porcentajes son un piso.

In [ ]:
hueco = metricas[(metricas["año"] == 2026) & metricas["categoria"].isin(NUCLEO) & (metricas["usuarios"] >= 150)]
hueco = hueco.set_index("tecnologia")[["categoria", "adopcion_pct", "interes_no_usuarios_pct", "retencion_pct"]]
hueco = hueco.sort_values("interes_no_usuarios_pct", ascending=False)

hueco.head(15).round(1)

### 3.5 Herramientas de IA

Serie corta: la encuesta pregunta por IA desde 2023 y cambió la pregunta todos los años (2023–2024 por producto, 2025 por modelo, 2026 por empresa). Por eso los modelos se agrupan por **proveedor**.

Acá la base es distinta al resto del notebook: el bloque de IA solo lo ve quien declara usar IA y llega al final de la encuesta, así que los porcentajes por proveedor y por agente son **entre los profesionales que respondieron ese bloque** (cuota entre usuarios de IA). El porcentaje general de uso de IA sale de otra pregunta (`AISelect`), que sí responde la mayoría.

In [ ]:
def usa_ia(año, respuesta):
    if año in (2023, 2024):
        return respuesta == "Yes"
    if año == 2025:
        return respuesta.str.startswith("Yes")
    return ~respuesta.str.contains("I don’t use AI tools|I don't use AI tools", regex=True)

filas = []
for año in (2023, 2024, 2025, 2026):
    sub = personas[(personas["año"] == año) & personas["es_profesional"] & personas["ai_select"].notna()]
    filas.append({"año": año, "respondieron": len(sub), "usa_ia_pct": usa_ia(año, sub["ai_select"]).mean() * 100})
uso_ia = pd.DataFrame(filas).set_index("año")

diario_2025 = personas[(personas["año"] == 2025) & personas["es_profesional"]]["ai_select"].dropna().eq("Yes, I use AI tools daily").mean() * 100
print(f"Profesionales que usan IA a diario en 2025: {diario_2025:.1f} %")
uso_ia.round(1)

In [ ]:
PROVEEDORES = ["OpenAI (ChatGPT / GPT)", "Anthropic (Claude)", "Google (Gemini)", "Microsoft (Bing / Phi)",
               "Meta (Llama)", "DeepSeek", "Mistral", "xAI (Grok)"]

def cuota_en_bloque(preguntas):
    # % de profesionales que marcó cada opción, sobre quienes respondieron ese bloque
    bloque = largo[(largo["relacion"] == "usa") & largo["pregunta"].isin(preguntas)]
    bloque = bloque.merge(personas.loc[personas["es_profesional"], ["año", "id"]], on=["año", "id"])
    base_bloque = bloque.groupby("año")["id"].nunique()
    cuota = bloque.groupby(["tecnologia", "año"], observed=True)["id"].nunique().unstack("año").div(base_bloque).mul(100)
    return cuota, base_bloque

cuota_modelos, base_modelos = cuota_en_bloque(["ia_chat", "ia_modelos"])
proveedores = cuota_modelos.loc[PROVEEDORES]
print("Profesionales que respondieron el bloque de modelos:", base_modelos.to_dict())
print("=== % que usó modelos o productos de cada proveedor (entre quienes respondieron el bloque) ===")
display(proveedores.round(1))
print("Nota: las series de Microsoft y Meta no son comparables (cambian de producto a modelo entre años); no se grafican.")

cuota_agentes, base_agentes = cuota_en_bloque(["ia_asistentes"])
print(f"=== Asistentes y agentes de código, 2026 (respondieron el bloque: {base_agentes[2026]:,}) ===")
display(cuota_agentes[2026].dropna().sort_values(ascending=False).head(8).round(1).to_frame("usa_pct"))

## 4. Perfiles y modalidades

### 4.1 Evolución de los perfiles

Los nombres de rol cambian todos los años y hasta 2022 la pregunta era de selección múltiple (después, única). Para comparar:
- cada rol se agrupa en una **familia** por palabras clave;
- cada persona reparte un peso de 1 entre los roles que marcó, así los años multi-select y single-select suman igual;
- se excluyen estudiantes, jubilados y "otros".

En 2017 la opción era "Web developer" sin distinguir front, back o full-stack, por eso el desarrollo web se muestra como una sola familia.

In [ ]:
FAMILIAS = [  # el orden importa: gana la primera que coincide
    ("Datos e IA", r"data scientist|machine learning|data or business analyst|data engineer|engineer, data|business intelligence"
                   r"|statistics or mathematics|^analyst$|ai/ml|developer, ai|applied scientist"),
    ("DevOps, cloud e infraestructura", r"devops|site reliability|cloud infrastructure|platform engineer|systems? administrator"),
    ("QA y testing", r"\bqa\b|quality assurance"),
    ("Mobile", r"mobile"),
    ("Desarrollo web", r"full-stack|back-end|front-end|^web developer$|enterprise level services"),
    ("Excluir", r"^student$|^retired$|^other"),
]

def familia_de(rol):
    for nombre, patron in FAMILIAS:
        if re.search(patron, rol.lower()):
            return nombre
    return "Otros roles"

roles = personas.loc[personas["es_profesional"] & personas["rol"].notna(), ["año", "id", "rol"]].copy()
roles["rol"] = roles["rol"].str.split(";")
roles = roles.explode("rol")
roles["rol"] = roles["rol"].str.strip()
roles["familia"] = roles["rol"].map({r: familia_de(r) for r in roles["rol"].unique()})
roles = roles[roles["familia"] != "Excluir"]
roles["peso"] = 1 / roles.groupby(["año", "id"])["rol"].transform("size")

perfiles = roles.pivot_table(index="familia", columns="año", values="peso", aggfunc="sum")
perfiles = perfiles.div(perfiles.sum()).mul(100)
perfiles.round(1)

In [ ]:
# Detalle dentro de web y dentro de datos/IA (participación sobre el total de roles, %)
detalle = {
    "Full-stack": r"full-stack", "Back-end": r"back-end|enterprise level services", "Front-end": r"front-end",
    "Científico de datos / ML": r"data scientist|machine learning|applied scientist|statistics or mathematics",
    "Analista de datos / BI": r"data or business analyst|business intelligence|^analyst$",
    "Ingeniero de datos": r"data engineer|engineer, data",
    "Ingeniero o desarrollador de IA": r"ai/ml|developer, ai",
}
total_por_año = roles.groupby("año")["peso"].sum()
tabla_detalle = pd.DataFrame({
    nombre: roles[roles["rol"].str.lower().str.contains(patron)].groupby("año")["peso"].sum() / total_por_año * 100
    for nombre, patron in detalle.items()
}).T
tabla_detalle.round(1)

### 4.2 Seniority

Años de experiencia profesional programando (en 2016 solo hay experiencia total). Tramos: junior (menos de 3 años), semi-senior (3 a 5), senior (6 o más).

In [ ]:
EXPERIENCIA_2016 = {"Less than 1 year": 0, "1 - 2 years": 1, "2 - 5 years": 3, "6 - 10 years": 6, "11+ years": 11}

def años_de_experiencia(texto):
    if texto.lower().startswith("less than"):
        return 0.0
    numero = re.search(r"\d+(\.\d+)?", texto)
    return float(numero.group()) if numero else np.nan

exp = personas[personas["es_profesional"]].copy()
exp["texto"] = exp["exp_pro"].where(exp["año"] != 2016, exp["exp_total"])
exp = exp.dropna(subset=["texto"])
exp["años_exp"] = np.where(exp["año"] == 2016, exp["texto"].map(EXPERIENCIA_2016),
                           exp["texto"].map({t: años_de_experiencia(t) for t in exp["texto"].unique()}))
exp = exp.dropna(subset=["años_exp"])
exp["seniority"] = pd.cut(exp["años_exp"], [-1, 2.99, 5.99, 100], labels=["Junior (<3)", "Semi-senior (3-5)", "Senior (6+)"])

seniority = (exp.groupby("año")["seniority"].value_counts(normalize=True).mul(100).unstack().round(1))
seniority["mediana_años"] = exp[exp["año"] >= 2019].groupby("año")["años_exp"].median()
seniority

### 4.3 Modalidad de trabajo

La pregunta existe en 2016, 2017, 2019 y 2022–2026 (no se hizo en 2018, 2020 ni 2021). Cambió de **frecuencia** ("¿cada cuánto trabajás remoto?", hasta 2019) a **esquema** ("remoto / híbrido / presencial", desde 2022). Equivalencias usadas:

| Categoría | Hasta 2019 | Desde 2022 |
|---|---|---|
| Remoto | tiempo completo remoto | Remote / Fully remote |
| Híbrido | al menos un día por semana remoto | Hybrid (todas las variantes) y "Your choice" |
| Presencial | nunca, casi nunca o pocos días al mes | In-person |

In [ ]:
MODALIDAD = {
    "Full-time remote": "Remoto", "Part-time remote": "Híbrido", "I rarely work remotely": "Presencial", "Never": "Presencial",
    "All or almost all the time (I'm full-time remote)": "Remoto",
    "More than half, but not all, the time": "Híbrido", "About half the time": "Híbrido",
    "Less than half the time, but at least one day each week": "Híbrido",
    "A few days each month": "Presencial", "Less than once per month / Never": "Presencial",
    "Fully remote": "Remoto", "Remote": "Remoto",
    "Hybrid (some remote, some in-person)": "Híbrido", "Hybrid (some remote, leans heavy to in-person)": "Híbrido",
    "Hybrid (some in-person, leans heavy to flexibility)": "Híbrido",
    "Your choice (very flexible, you can come in when you want or just as needed)": "Híbrido",
    "Full in-person": "Presencial", "In-person": "Presencial",
}
trabajo = personas[personas["es_profesional"] & personas["remoto"].notna()].copy()
trabajo["modalidad"] = trabajo["remoto"].map(MODALIDAD)
print("Respuestas sin equivalencia (se excluyen):", trabajo.loc[trabajo["modalidad"].isna(), "remoto"].value_counts().to_dict())

modalidad = (trabajo.dropna(subset=["modalidad"]).groupby("año")["modalidad"].value_counts(normalize=True)
             .mul(100).unstack()[["Presencial", "Híbrido", "Remoto"]])
modalidad.round(1)

## 5. Cruce con las ofertas de nextgig (2026)

Conecta la tendencia con el dataset principal del equipo. Para cada tecnología se cruzan dos cosas:
- **demanda hoy:** porcentaje de ofertas que la piden (`data/processed/ofertas_tech_limpio.csv`, ofertas con skills informadas);
- **tendencia:** pendiente de adopción en la encuesta (puntos por año sobre todos los años disponibles).

Los niveles de las dos fuentes no son comparables entre sí (una oferta lista unas 8 skills; un desarrollador marca todo lo que usó en el año), y las ofertas son solo de los 5 roles del caso. Lo que sí se puede leer es la combinación: qué tecnologías son pedidas **y** vienen en alza.

In [ ]:
ofertas = pd.read_csv(f"{RUTA_PROCESSED}/ofertas_tech_limpio.csv")
ofertas = ofertas[ofertas["tiene_skills"]].copy()
skills = ofertas["skills"].str.lower()

PATRONES = {
    "Python": r"python", "SQL": r"\bsql\b", "JavaScript": r"javascript|\bjs\b", "TypeScript": r"typescript",
    "Java": r"\bjava\b", "C#": r"c#", "C++": r"c\+\+", "Go": r"\bgo\b|golang", "R": r"(?:^|\|)r(?:\||$)", "Bash/Shell": r"bash|shell",
    "PHP": r"\bphp\b", "Rust": r"\brust\b", "Kotlin": r"kotlin", "HTML/CSS": r"\bhtml|\bcss",
    "AWS": r"\baws\b|amazon web services", "Microsoft Azure": r"azure", "Google Cloud": r"\bgcp\b|google cloud",
    "Docker": r"docker", "Kubernetes": r"kubernetes|\bk8s\b", "Terraform": r"terraform", "Ansible": r"ansible",
    "React": r"\breact", "Angular": r"angular", "Node.js": r"node\.?js|\bnode\b", "Vue.js": r"\bvue", "Next.js": r"next\.?js",
    "Spring": r"spring", "Django": r"django", "FastAPI": r"fastapi", "ASP.NET": r"asp\.net",
    "PostgreSQL": r"postgres", "MySQL": r"mysql", "SQL Server": r"sql server|mssql", "MongoDB": r"mongo",
    "Redis": r"redis", "Snowflake": r"snowflake", "BigQuery": r"bigquery", "Elasticsearch": r"elasticsearch",
}
cruce = pd.DataFrame({
    "categoria": categoria.reindex(list(PATRONES)),
    "adopcion_so_2026": adopcion[2026].reindex(list(PATRONES)),
    "ofertas_pct": [skills.str.contains(patron, regex=True).mean() * 100 for patron in PATRONES.values()],
})
cruce["pendiente_pp_por_año"] = tendencia["pendiente_pp_por_año"].reindex(cruce.index)
cruce["delta_2018_2026_pp"] = tendencia["delta_2018_2026_pp"].reindex(cruce.index)
cruce = cruce.dropna(subset=["adopcion_so_2026", "pendiente_pp_por_año"]).sort_values("ofertas_pct", ascending=False)

print(f"Ofertas con skills: {len(ofertas)}")
cruce.round(1)

## 6. Visualizaciones

Todas se guardan en `images/` como PNG. Convenciones: un solo eje por gráfico, color fijo por tecnología, valores vacíos donde la encuesta no preguntó.

In [ ]:
SUPERFICIE, TINTA, TINTA_2, MUDO, GRILLA, EJE = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
SERIE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
AZUL, ROJO, NEUTRO = "#2a78d6", "#e34948", "#f0efec"
SECUENCIAL = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]

plt.rcParams.update({
    "figure.facecolor": SUPERFICIE, "axes.facecolor": SUPERFICIE, "savefig.facecolor": SUPERFICIE,
    "font.family": "sans-serif", "font.size": 10, "text.color": TINTA, "axes.labelcolor": TINTA_2,
    "axes.edgecolor": EJE, "axes.linewidth": 1, "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRILLA, "grid.linewidth": 1,
    "axes.axisbelow": True, "xtick.color": MUDO, "ytick.color": MUDO, "xtick.labelcolor": TINTA_2,
    "ytick.labelcolor": TINTA_2, "ytick.left": False, "xtick.bottom": False, "legend.frameon": False,
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "lines.solid_joinstyle": "round",
})

def titular(fig, titulo, subtitulo, y=0.98):
    fig.text(0.02, y, titulo, fontsize=13, fontweight="bold", ha="left", va="top", color=TINTA)
    fig.text(0.02, y - 0.055, subtitulo, fontsize=9.5, ha="left", va="top", color=TINTA_2)

def fuente(fig, texto="Fuente: Stack Overflow Developer Survey 2016-2026, desarrolladores profesionales."):
    fig.text(0.02, 0.012, texto, fontsize=8, ha="left", va="bottom", color=MUDO)

def guardar(fig, nombre):
    fig.savefig(f"{RUTA_IMAGENES}/{nombre}", dpi=200)
    plt.show()

def etiquetas_al_final(ax, finales, separacion):
    # etiqueta cada serie en su último punto; si se pisan, las separa hacia arriba sin cruzarlas
    y_previo = -np.inf
    for nombre, (x, y) in sorted(finales.items(), key=lambda kv: kv[1][1]):
        y_texto = max(y, y_previo + separacion)
        ax.annotate(f"{nombre}  {y:.0f} %", (x, y), xytext=(x + 0.25, y_texto), fontsize=9, color=TINTA, va="center",
                    arrowprops=dict(arrowstyle="-", color=EJE, lw=0.8) if abs(y_texto - y) > 0.3 * separacion else None)
        y_previo = y_texto

In [ ]:
# 6.1 Subidas y bajadas 2018 -> 2026, en dos columnas
from matplotlib.patches import FancyBboxPatch

COLUMNAS = [
    ("Suben", "Cada vez más desarrolladores las usan", AZUL, "▲", subidas.sort_values("delta_2018_2026_pp", ascending=False)),
    ("Bajan", "Cada vez menos desarrolladores las usan", ROJO, "▼", bajadas.sort_values("delta_2018_2026_pp")),
]

fig, ax = plt.subplots(figsize=(10, 6.6))
fig.subplots_adjust(left=0.03, right=0.97, top=0.82, bottom=0.07)
ax.set_xlim(0, 2); ax.set_ylim(0, 1); ax.axis("off"); ax.grid(False)
margen = 0.04
alto_fila = 0.068
for k, (titulo, descripcion, color, flecha, grupo) in enumerate(COLUMNAS):
    x0, y0 = k + margen, 0.02
    ancho, alto = 1 - 2 * margen, 0.96
    ax.add_patch(FancyBboxPatch((x0, y0), ancho, alto, boxstyle="round,pad=0,rounding_size=0.03", facecolor=color, alpha=0.10, edgecolor="none"))
    ax.add_patch(FancyBboxPatch((x0, y0), ancho, alto, boxstyle="round,pad=0,rounding_size=0.03", facecolor="none", edgecolor=color, linewidth=1.5))
    ax.text(x0 + 0.05, y0 + alto - 0.06, titulo, fontsize=14, fontweight="bold", color=color, va="center")
    ax.text(x0 + 0.05, y0 + alto - 0.12, descripcion, fontsize=9, color=TINTA_2, va="center")
    for i, (tec, fila) in enumerate(grupo.iterrows()):
        y = y0 + alto - 0.21 - i * alto_fila
        inicio, fin = tendencia.loc[tec, ["adopcion_2018", "adopcion_2026"]]
        ax.text(x0 + 0.05, y, tec, fontsize=10.5, color=TINTA, va="center")
        ax.text(x0 + ancho - 0.05, y, f"{inicio:.0f} → {fin:.0f} %   {flecha}", fontsize=10.5, color=TINTA_2, va="center", ha="right")
titular(fig, "Qué ganó y qué perdió lugar entre 2018 y 2026",
        "De cada 100 desarrolladores profesionales, cuántos usaban cada tecnología en 2018 y cuántos en 2026.\nPor ejemplo, TypeScript pasó de 19 a 49 de cada 100.")
fuente(fig)
guardar(fig, "03_subidas_bajadas.png")


In [ ]:
# 6.2 IA: adopción general y proveedores
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 5.2), gridspec_kw={"width_ratios": [1, 1.7]})
fig.subplots_adjust(left=0.06, right=0.80, top=0.74, bottom=0.17, wspace=0.22)

ax1.bar(uso_ia.index, uso_ia["usa_ia_pct"], width=0.5, color=AZUL)
for año, v in uso_ia["usa_ia_pct"].items():
    ax1.text(año, v + 1.5, f"{v:.0f} %", ha="center", fontsize=10, color=TINTA)
ax1.set_ylim(0, 100); ax1.set_xticks(uso_ia.index); ax1.set_yticks(range(0, 101, 25))
ax1.set_yticklabels([f"{v} %" for v in range(0, 101, 25)])
ax1.set_title("Usa herramientas de IA en su trabajo", fontsize=10, loc="left", color=TINTA_2)

GRAFICAR = ["OpenAI (ChatGPT / GPT)", "Anthropic (Claude)", "Google (Gemini)", "DeepSeek"]
finales = {}
for color, proveedor in zip(SERIE, GRAFICAR):
    serie = proveedores.loc[proveedor].dropna()
    ax2.plot(serie.index, serie.values, color=color, label=proveedor, marker="o", markersize=5)
    ax2.plot(serie.index[-1], serie.values[-1], "o", color=color, markersize=8, markeredgecolor=SUPERFICIE, markeredgewidth=2)
    finales[proveedor.split(" (")[0]] = (serie.index[-1], serie.values[-1])
etiquetas_al_final(ax2, finales, separacion=5.5)
ax2.set_ylim(0, 100); ax2.set_xlim(2022.8, 2026.15); ax2.set_xticks([2023, 2024, 2025, 2026])
ax2.set_yticks(range(0, 101, 25)); ax2.set_yticklabels([f"{v} %" for v in range(0, 101, 25)])
ax2.set_title("Proveedor de los modelos que usa (entre quienes usan IA)", fontsize=10, loc="left", color=TINTA_2)
ax2.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=9, handlelength=1.4, columnspacing=1.6)

titular(fig, "La IA pasó de novedad a herramienta de trabajo en tres años",
        "Desarrolladores profesionales. La pregunta cambió cada año (producto en 2023-24, modelo en 2025, empresa en 2026):\nla tendencia general es firme, los saltos de un año a otro por proveedor son orientativos", y=0.975)
fuente(fig, "Fuente: Stack Overflow Developer Survey 2023-2026, desarrolladores profesionales.")
guardar(fig, "03_ia_adopcion.png")

In [ ]:
# 6.3 Hueco de aprendizaje 2026
top_hueco = hueco.head(15).iloc[::-1]

fig, ax = plt.subplots(figsize=(10, 6.6))
fig.subplots_adjust(left=0.16, right=0.93, top=0.85, bottom=0.08)
ax.barh(range(len(top_hueco)), top_hueco["interes_no_usuarios_pct"], height=0.6, color=AZUL)
for y, (tec, fila) in enumerate(top_hueco.iterrows()):
    ax.text(fila["interes_no_usuarios_pct"] + 0.3, y,
            f"{fila['interes_no_usuarios_pct']:.0f} %   ·   hoy la usa el {fila['adopcion_pct']:.0f} %", va="center", fontsize=9, color=TINTA)
ax.set_yticks(range(len(top_hueco))); ax.set_yticklabels(top_hueco.index, color=TINTA)
ax.set_xlim(0, top_hueco["interes_no_usuarios_pct"].max() * 1.35); ax.set_xticks([]); ax.grid(False)
ax.spines["bottom"].set_visible(False)
titular(fig, "Qué quieren aprender quienes todavía no lo usan",
        "% de profesionales que NO usa la tecnología y quiere trabajar con ella el próximo año (2026)")
fuente(fig, "Fuente: Stack Overflow Developer Survey 2026, desarrolladores profesionales.")
guardar(fig, "03_hueco_aprendizaje.png")

In [ ]:
# 6.4 Perfiles (sin desarrollo web, que domina la escala y va en el subtítulo)
FAMILIAS_GRAFICO = ["Datos e IA", "DevOps, cloud e infraestructura", "Mobile", "QA y testing"]

fig, ax = plt.subplots(figsize=(10, 5.4))
fig.subplots_adjust(left=0.07, right=0.76, top=0.78, bottom=0.17)
finales = {}
for color, familia in zip(SERIE, FAMILIAS_GRAFICO):
    serie = perfiles.loc[familia].dropna()
    ax.plot(serie.index, serie.values, color=color, label=familia)
    ax.plot(serie.index[-1], serie.values[-1], "o", color=color, markersize=8, markeredgecolor=SUPERFICIE, markeredgewidth=2)
    finales[familia] = (serie.index[-1], serie.values[-1])
etiquetas_al_final(ax, finales, separacion=0.75)
ax.axvline(2022.5, color=EJE, linewidth=1)
ax.text(2022.6, 11.7, "desde 2023 se elige\nun solo rol", fontsize=8, color=MUDO, va="top")
ax.set_xlim(2015.7, 2026.3); ax.set_ylim(0, 12); ax.set_xticks(AÑOS)
ax.set_yticks(range(0, 13, 3)); ax.set_yticklabels([f"{v} %" for v in range(0, 13, 3)])
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.08), ncol=4, fontsize=9, handlelength=1.4)
web = perfiles.loc["Desarrollo web"]
titular(fig, "Datos e IA es la única familia de perfiles que crece desde 2023",
        f"Participación de cada familia sobre el total de roles declarados por profesionales.\n"
        f"Desarrollo web (no graficado) pasó de {web[2016]:.0f} % en 2016 a {web[2026]:.0f} % en 2026", y=0.975)
fuente(fig)
guardar(fig, "03_perfiles_evolucion.png")

In [ ]:
# 6.5 Cruce simple: cuatro grupos según lo que piden hoy las empresas y si la tecnología crece entre desarrolladores
from matplotlib.patches import FancyBboxPatch

UMBRAL_OFERTAS = 10     # % de ofertas: desde ahí se considera "muy pedida"
UMBRAL_TENDENCIA = 1.0  # puntos porcentuales por año: por encima se considera "crece"
MAX_POR_CAJA = 8

pedida = cruce["ofertas_pct"] >= UMBRAL_OFERTAS
crece = cruce["pendiente_pp_por_año"] > UMBRAL_TENDENCIA
# (título, descripción, color, fila, columna, filtro); fila 0 = muy pedidas, columna 0 = crece
CAJAS = [
    ("Apuesta segura", "Se pide mucho y además crece", AZUL, 0, 0, pedida & crece),
    ("Se pide, pero no crece", "Siguen dando trabajo, sin ganar terreno", "#eda100", 0, 1, pedida & ~crece),
    ("Oportunidad", "Se pide poco todavía, pero crece", "#1baf7a", 1, 0, ~pedida & crece),
    ("Baja prioridad", "Se pide poco y no crece", MUDO, 1, 1, ~pedida & ~crece),
]

fig, ax = plt.subplots(figsize=(10, 7.4))
fig.subplots_adjust(left=0.03, right=0.97, top=0.85, bottom=0.09)
ax.set_xlim(0, 2); ax.set_ylim(0, 2); ax.axis("off"); ax.grid(False)
margen = 0.04
for titulo, descripcion, color, fila, columna, filtro in CAJAS:
    x0, y0 = columna + margen, (1 - fila) + margen
    ancho = alto = 1 - 2 * margen
    ax.add_patch(FancyBboxPatch((x0, y0), ancho, alto, boxstyle="round,pad=0,rounding_size=0.03", facecolor=color, alpha=0.10, edgecolor="none"))
    ax.add_patch(FancyBboxPatch((x0, y0), ancho, alto, boxstyle="round,pad=0,rounding_size=0.03", facecolor="none", edgecolor=color, linewidth=1.5))
    ax.text(x0 + 0.05, y0 + alto - 0.07, titulo, fontsize=13, fontweight="bold", color=color, va="center")
    ax.text(x0 + 0.05, y0 + alto - 0.15, descripcion, fontsize=9, color=TINTA_2, va="center")
    grupo = cruce[filtro].sort_values("ofertas_pct", ascending=False)
    for k, (tec, fila_tec) in enumerate(grupo.head(MAX_POR_CAJA).iterrows()):
        y = y0 + alto - 0.26 - k * 0.075
        ax.text(x0 + 0.05, y, tec, fontsize=10.5, color=TINTA, va="center")
        ax.text(x0 + ancho - 0.05, y, f"{fila_tec['ofertas_pct']:.0f} %", fontsize=10.5, color=TINTA_2, va="center", ha="right")
    if len(grupo) > MAX_POR_CAJA:
        ax.text(x0 + 0.05, y0 + 0.04, f"y {len(grupo) - MAX_POR_CAJA} más", fontsize=8.5, color=MUDO, va="center")
titular(fig, "Python y cloud son la apuesta más segura para el bootcamp",
        "Cada tecnología se agrupa según cuántas ofertas la piden hoy y si cada año la usan más desarrolladores.\nEl número es el % de ofertas que la piden.")
fuente(fig, f"Fuentes: nextgig, {len(ofertas)} ofertas con skills (junio 2026), y Stack Overflow Developer Survey 2016-2026 (profesionales).\nMuy pedida: {UMBRAL_OFERTAS} % de ofertas o más. Crece: más de {UMBRAL_TENDENCIA:.0f} punto porcentual de adopción por año.")
guardar(fig, "03_cruce_ofertas.png")


## 7. Guardar datasets procesados

- `so_survey_adopcion_anual.csv`: una fila por año × tecnología, con la base de respuestas y los indicadores (profesionales y total de encuestados). Es la tabla para cruzar con los puntos 1, 2 y 4.
- `so_survey_adopcion_matriz.csv`: la misma adopción de profesionales en formato tecnología × año.
- `so_survey_tendencias.csv`: resumen por tecnología (primer año, deltas, pendiente).
- `so_survey_panel_limpio.parquet`: panel largo completo (respondente × tecnología).

In [ ]:
columnas_todos = {"usuarios": "usuarios_todos", "base": "base_todos", "adopcion_pct": "adopcion_todos_pct"}
anual = metricas.merge(metricas_todos.rename(columns=columnas_todos)[["año", "tecnologia"] + list(columnas_todos.values())],
                       on=["año", "tecnologia"], how="left")
anual = anual[["año", "tecnologia", "categoria", "usuarios", "base", "adopcion_pct", "interes_no_usuarios_pct",
               "retencion_pct", "usuarios_todos", "base_todos", "adopcion_todos_pct"]].sort_values(["tecnologia", "año"])
columnas_pct = [c for c in anual.columns if c.endswith("_pct")]
anual[columnas_pct] = anual[columnas_pct].round(2)

anual.to_csv(f"{RUTA_PROCESSED}/so_survey_adopcion_anual.csv", index=False)
adopcion.round(2).to_csv(f"{RUTA_PROCESSED}/so_survey_adopcion_matriz.csv")
tendencia.round(2).to_csv(f"{RUTA_PROCESSED}/so_survey_tendencias.csv")
panel.to_parquet(f"{RUTA_PROCESSED}/so_survey_panel_limpio.parquet", index=False, compression="zstd")

print(f"Adopción anual: {len(anual):,} filas | Matriz: {adopcion.shape} | Tendencias: {len(tendencia)} | Panel: {len(panel):,} filas")